# Download data from yfinance

In [ ]:
# !pip install yfinance
import pandas as pd
import yfinance as yf
import numpy as np
from datetime import datetime
from sklearn.decomposition import PCA

# ===== Step 1. Define tickers =====
tickers = [
    "^GSPC",   # S&P 500 Index
    "^VIX",    # VIX (implied volatility)
    "XLK",     # Technology sector ETF
    "XLE",     # Energy sector ETF
    "XLF",     # Financial sector ETF
    "^TNX",    # 10Y Treasury Yield
    "^IRX",    # 3M Treasury Bill
    "^NYICDX", # Credit default swap index
    "ES=F"     # S&P 500 E-mini Futures
]

start, end = "2020-09-01", "2025-09-01"

# ===== Step 2. Download data and compute log returns =====
all_data = {}
for t in tickers:
    df = yf.download(t, start=start, end=end, auto_adjust=False)
    df["LogReturn"] = np.log(df["Adj Close"] / df["Adj Close"].shift(1))
    df = df.rename(columns=lambda c: f"{t}_{c.replace(' ', '')}")
    all_data[t] = df

merged_df = pd.concat(all_data.values(), axis=1, join="outer").sort_index()

# ===== Step 3. Build feature matrix =====
features = pd.DataFrame(index=merged_df.index)

def safe_pick(df, like_str, new_name):
    cols = df.filter(like=like_str).columns
    if len(cols) > 0:
        features[new_name] = df[cols[0]]

# Market return
safe_pick(merged_df, "GSPC_LogReturn", "SP500_Ret")

# Sector returns
for ticker in ["XLK", "XLE", "XLF"]:
    safe_pick(merged_df, f"{ticker}_LogReturn", f"{ticker}_Ret")

# VIX
safe_pick(merged_df, "VIX_Close", "VIX")

# Interest rates
safe_pick(merged_df, "TNX_Close", "TNX")
safe_pick(merged_df, "IRX_Close", "IRX")

# NYICDX return
safe_pick(merged_df, "NYICDX_LogReturn", "NYICDX_Ret")

# Volume + Amihud illiquidity
for ticker in ["XLK", "XLE", "XLF"]:
    vol_col = merged_df.filter(like=f"{ticker}_Volume").columns
    ret_col = merged_df.filter(like=f"{ticker}_LogReturn").columns
    if len(vol_col) > 0 and len(ret_col) > 0:
        features[f"{ticker}_Volume"] = merged_df[vol_col[0]]
        features[f"{ticker}_Amihud"] = merged_df[ret_col[0]].abs() / merged_df[vol_col[0]]

# ===== Step 3.5. Volatility proxies & PCA combine =====
sp500_ret_col = merged_df.filter(like="GSPC_LogReturn").columns
if len(sp500_ret_col) > 0:
    sp500_ret = merged_df[sp500_ret_col[0]]

    # Compute temporary RealVol and RangeVol
    real_vol = sp500_ret.rolling(5).apply(lambda x: np.sqrt(np.mean(x**2)))

    high_col = merged_df.filter(like="GSPC_High").columns
    low_col = merged_df.filter(like="GSPC_Low").columns
    close_col = merged_df.filter(like="GSPC_Close").columns

    if len(high_col) > 0 and len(low_col) > 0 and len(close_col) > 0:
        range_vol = (merged_df[high_col[0]] - merged_df[low_col[0]]) / merged_df[close_col[0]]

        # Combine RealVol + RangeVol into a single SPV factor using PCA
        pca_data = pd.concat([real_vol, range_vol], axis=1).dropna()
        pca_data.columns = ["RealVol", "RangeVol"]

        pca = PCA(n_components=1)
        spv_factor = pca.fit_transform(pca_data)
        spv_series = pd.Series(spv_factor.flatten(), index=pca_data.index)

        # Final SPV column
        features["SPV"] = spv_series

# ===== Step 4. Pricing Error =====
spy = yf.Ticker("SPY")
div_yield = spy.info.get("dividendYield", 0.015)  # fallback = 1.5%

spot = merged_df.filter(like="GSPC_AdjClose").iloc[:, 0]
futures = merged_df.filter(like="ES=F_AdjClose").iloc[:, 0]
rate = merged_df.filter(like="IRX_AdjClose").iloc[:, 0]

expiry = datetime(2024, 6, 21)

df_pricing = pd.DataFrame({
    "Spot": spot,
    "Futures": futures,
    "IRX": rate
}).dropna()

df_pricing["T_minus_t"] = (expiry - df_pricing.index).days / 365
df_pricing["r"] = df_pricing["IRX"] / 100

df_pricing["PricingError"] = np.log(df_pricing["Futures"]) - np.log(
    df_pricing["Spot"] * np.exp((df_pricing["r"] - div_yield) * df_pricing["T_minus_t"])
)

features["PricingError"] = df_pricing["PricingError"]

# ===== Step 5. Clean & standardize =====
features = features.dropna()
features = (features - features.mean()) / features.std()

print("Final feature matrix shape:", features.shape)
print(features.head())
features.to_csv("causal_feature_matrix.csv")


Certification

In [ ]:
# -*- coding: utf-8 -*-
"""
This script constructs a sector-level DAG (including latent variables),
checks identifiability for known graphs, and performs 'unknown graph'
identification directly from observed data (Tramontano et al., 2024).

Requirements:
- Python 3.10+
- pandas
- numpy
- networkx==3.3
- scikit-learn
"""

import networkx as nx
import pandas as pd
import numpy as np
import time
from sklearn.decomposition import FastICA
from networkx.algorithms.dag import topological_sort
!pip uninstall -y networkx
!pip install networkx==3.3


# ---------------------------------------------------------------------
# Load dataset and create lagged variables
# ---------------------------------------------------------------------
CSV_PATH = "causal_feature_matrix.csv"  # change if needed
df_raw = pd.read_csv(CSV_PATH, parse_dates=["Date"], index_col="Date")


# Create *_t and *_t-1 columns
df = df_raw.copy()
orig_cols = list(df_raw.columns)
for col in orig_cols:
    cur = f"{col}_t" if not col.endswith("_t") else col
    lag = f"{col}_t-1" if not col.endswith("_t-1") else col
    df[cur] = df_raw[col].values
    df[lag] = df_raw[col].shift(1).values
df = df.dropna()


# -----------------------------------------------------------
# Complete name_map (including returns, macro variables,
# sector liquidity, and volatility proxies)
# -----------------------------------------------------------
name_map = {
    # === Market Returns ===
    "SP500_Ret_t": "SP500_t",
    "SP500_Ret_t-1": "SP500_t-1",

    # === Sector Returns ===
    "XLK_Ret_t": "XLK_t",
    "XLK_Ret_t-1": "XLK_t-1",
    "XLE_Ret_t": "XLE_t",
    "XLE_Ret_t-1": "XLE_t-1",
    "XLF_Ret_t": "XLF_t",
    "XLF_Ret_t-1": "XLF_t-1",

    # === Macro & Credit Market Variables ===
    "NYICDX_Ret_t": "NYICDX_t",
    "NYICDX_Ret_t-1": "NYICDX_t-1",
    "VIX_t": "VIX_t",
    "VIX_t-1": "VIX_t-1",
    "TNX_t": "TNX_t",
    "TNX_t-1": "TNX_t-1",
    "IRX_t": "IRX_t",
    "IRX_t-1": "IRX_t-1",

    # === Proxy Variables ===
    "PricingError_t": "PricingError_t",
    "PricingError_t-1": "PricingError_t-1",

    # === Sector Trading Volume ===
    "XLK_Volume_t": "XLK_Volume_t",
    "XLK_Volume_t-1": "XLK_Volume_t-1",
    "XLE_Volume_t": "XLE_Volume_t",
    "XLE_Volume_t-1": "XLE_Volume_t-1",
    "XLF_Volume_t": "XLF_Volume_t",
    "XLF_Volume_t-1": "XLF_Volume_t-1",

    # === Sector Liquidity (Amihud Measure) ===
    "XLK_Amihud_t": "XLK_Amihud_t",
    "XLK_Amihud_t-1": "XLK_Amihud_t-1",
    "XLE_Amihud_t": "XLE_Amihud_t",
    "XLE_Amihud_t-1": "XLE_Amihud_t-1",
    "XLF_Amihud_t": "XLF_Amihud_t",
    "XLF_Amihud_t-1": "XLF_Amihud_t-1",

    # === Volatility Proxy (SPV) ===
     "SPV_t": "SPV_t",
     "SPV_t-1": "SPV_t-1",
}

df = df.rename(columns=name_map)


# ---------------------------------------------------------------------
# DAG construction (feedback edges removed)
# ---------------------------------------------------------------------
def build_graph_core_sector():
    """
    Constructs the core DAG containing latent variables and observed
    sector-level nodes. Returns (graph, latent_nodes, sector_vol, sector_ami).
    """
    G = nx.DiGraph()

    # ----------- Nodes -----------
    latent_nodes = ["L_micro_t-1", "L_sentiment_t-1"]  # latent variables
    observed_core = ["NYICDX", "IRX", "TNX", "VIX", "SP500", "XLK", "XLF", "XLE"]
    proxies = ["OpenInterest", "PricingError", "SPV"]

    # sector-level proxies
    sector_vol = [f"{s}_Volume" for s in ["XLK", "XLF", "XLE"]]
    sector_ami = [f"{s}_Amihud" for s in ["XLK", "XLF", "XLE"]]

    observed_t1 = [f"{v}_t-1" for v in observed_core + proxies + sector_vol + sector_ami]
    observed_t = [f"{v}_t" for v in observed_core + proxies + sector_vol + sector_ami]

    G.add_nodes_from(latent_nodes + observed_t1 + observed_t)

    # ----------- Latent variables -----------
    G.add_edge("L_micro_t-1", "IRX_t")
    G.add_edge("L_micro_t-1", "TNX_t")
    G.add_edge("L_micro_t-1", "PricingError_t")

    G.add_edge("L_sentiment_t-1", "VIX_t")
    for vol in sector_vol:
        G.add_edge("L_sentiment_t-1", f"{vol}_t")
    for ami in sector_ami:
        G.add_edge("L_sentiment_t-1", f"{ami}_t")

    # ----------- Macro factors → SP500 -----------
    for m in ["NYICDX", "IRX", "TNX", "VIX"]:
        G.add_edge(f"{m}_t-1", "SP500_t")
    for ind in ["XLK", "XLF", "XLE"]:
        G.add_edge(f"{ind}_t-1", "SP500_t")

    # ----------- Proxy variables -----------
    G.add_edge("SP500_t-1", "OpenInterest_t")
    G.add_edge("SP500_t-1", "PricingError_t")

    # sector-level proxy connections (latent/proxy → observed)
    for vol in sector_vol:
        G.add_edge("SPV_t-1", f"{vol}_t")
    for ami in sector_ami:
        G.add_edge("SPV_t-1", f"{ami}_t")

    # ----------- Persistence (auto-regressive lags) -----------
    for v in observed_core + proxies + sector_vol + sector_ami:
        G.add_edge(f"{v}_t-1", f"{v}_t")

    assert nx.is_directed_acyclic_graph(G), "Graph must be a DAG"
    return G, latent_nodes, sector_vol, sector_ami


# ---------------------------------------------------------------------
# Identifiability functions (as before)
# ---------------------------------------------------------------------
def _to_indexed_graph_and_edge(G, edge_tuple):
    top_order = list(topological_sort(G))
    name_to_idx = {name: i for i, name in enumerate(top_order)}
    G_indexed = nx.relabel_nodes(G, mapping=name_to_idx, copy=True)
    edge_idx = (name_to_idx[edge_tuple[0]], name_to_idx[edge_tuple[1]])
    return G_indexed, edge_idx, top_order


def check_pair_tce_id_named(G, edge, latent_names):
    start = time.time()
    G_idx, edge_idx, _ = _to_indexed_graph_and_edge(G, edge)
    latent = len(latent_names)
    id_known, id_unknown = True, True

    for k in range(latent):
        de_k = nx.descendants(G_idx, k)
        if edge_idx[1] in de_k:
            other_children = set(G_idx.successors(k)) - {edge_idx[0]}
            if other_children:
                id_known = False
                id_unknown = False
    return id_known, id_unknown, time.time() - start


def full_graph_id_named(G, latent_names):
    start = time.time()
    G_idx = nx.relabel_nodes(G, mapping={n: i for i, n in enumerate(list(topological_sort(G)))}, copy=True)
    latent = len(latent_names)
    id_known, id_unknown = True, True

    for k in range(latent):
        de_k = nx.descendants(G_idx, k)
        children = list(G_idx.successors(k))
        if len(children) > 1:
            for j in children:
                de_j = nx.descendants(G_idx, j) | {j}
                if de_j == de_k:
                    id_known = False
                    id_unknown = False
    return id_known, id_unknown, time.time() - start


# ---------------------------------------------------------------------
# Build the DAG first
# ---------------------------------------------------------------------
G_new, latent_new, sector_vol, sector_ami = build_graph_core_sector()


# ---------------------------------------------------------------------
# Generate edges_to_check for ALL node pairs (Known Graph)
# ---------------------------------------------------------------------
all_nodes = list(G_new.nodes())
edges_to_check = [(a, b) for a in all_nodes for b in all_nodes if a != b]

results = []
for e in edges_to_check:
    idk, idu, ttime = check_pair_tce_id_named(G_new, e, latent_new)
    results.append({
        "edge": f"{e[0]} -> {e[1]}",
        "id_known": idk,
        "time_sec": round(ttime, 6)
    })

df_results = pd.DataFrame(results)
global_idk, global_idu, global_t = full_graph_id_named(G_new, latent_new)

print("\n=== Edge-level Identification Results (Known Graph, FULL) ===")
print(df_results.to_string(index=False))
print("\n=== Global Graph Identification Summary (Known Graph) ===")
print(f"id_known: {global_idk}, time_sec: {round(global_t,6)}")


# ---------------------------------------------------------------------
# Count number of True / False for known-graph identifiability
# ---------------------------------------------------------------------
true_count_known = df_results["id_known"].sum()
false_count_known = len(df_results) - true_count_known

print("\n=== Edge-level Identification Summary (Known Graph) ===")
print(f"Total edges checked: {len(df_results)}")
print(f"Identifiable (True): {true_count_known}")
print(f"Not identifiable (False): {false_count_known}")





# ---------------------------------------------------------------------
# Unknown Graph Certification (from observed data)
# ---------------------------------------------------------------------
def unknown_graph_certification_from_data(data_table,
                                          threshold=1e-2,
                                          edges_to_check=None):
    """
    Performs unknown-graph identifiability test using estimated mixing matrix B'
    from FastICA, following Tramontano et al. (2024).
    Returns global and edge-level identifiability results.
    """

    start = time.time()
    X = np.asarray(data_table)
    ica = FastICA(n_components=X.shape[1], random_state=0, max_iter=2000)
    S_ = ica.fit_transform(X)
    B_est = np.linalg.pinv(ica.mixing_)

    # ----- Build adjacency (approximate inferred graph) -----
    A_est = (np.abs(B_est) > threshold).astype(int)
    np.fill_diagonal(A_est, 0)
    G_est = nx.DiGraph(A_est)

    # ----- Global unknown-graph test -----
    supports = [set(np.where(np.abs(B_est[:, j]) > threshold)[0]) for j in range(B_est.shape[1])]
    unique_supports = len(supports) == len({frozenset(s) for s in supports})
    id_unknown_global = unique_supports

    # ----- Edge-level unknown-graph test -----
    results = []
    if edges_to_check is not None:
        colnames = list(data_table.columns)
        for e in edges_to_check:
            src, tgt = e
            if src not in colnames or tgt not in colnames:
                results.append({
                    "edge": f"{src} -> {tgt}",
                    "id_unknown": None,
                    "note": "missing_column"
                })
                continue

            j_idx = colnames.index(src)
            i_idx = colnames.index(tgt)

            # Simple pairwise check: if columns i and j have identical supports, mark non-identifiable
            same_support = supports[i_idx] == supports[j_idx]
            id_unknown_pair = not same_support

            results.append({
                "edge": f"{src} -> {tgt}",
                "id_unknown": id_unknown_pair,

            })
    df_edge = pd.DataFrame(results) if results else None

    total_time = time.time() - start
    return id_unknown_global, df_edge, total_time


# ---------------------------------------------------------------------
# Generate edges_to_check for ALL node pairs (Unknown Graph)
# ---------------------------------------------------------------------
data_cols = [c for c in df.columns if c.endswith("_t") or c.endswith("_t-1")]
data_for_unknown = df[data_cols].copy()

edges_for_unknown = [(a, b) for a in data_cols for b in data_cols if a != b]

id_unknown_u, df_edge_unknown, time_u = unknown_graph_certification_from_data(
    data_for_unknown, threshold=0.05, edges_to_check=edges_for_unknown)

print("\n=== Edge-level Identification Results (Unknown Graph, FULL) ===")
if df_edge_unknown is not None:
    print(df_edge_unknown.to_string(index=False))
else:
    print("No edge-level results (no matching columns found).")

print("\n=== Global Graph Identification Summary (Unknown Graph from Data) ===")
print(f"id_unknown: {id_unknown_u}, time_sec: {round(time_u, 4)}")



# ---------------------------------------------------------------------
# Count number of True / False for unknown-graph identifiability
# ---------------------------------------------------------------------
if df_edge_unknown is not None and "id_unknown" in df_edge_unknown.columns:
    true_count_unknown = df_edge_unknown["id_unknown"].sum()
    false_count_unknown = len(df_edge_unknown) - true_count_unknown

    print("\n=== Edge-level Identification Summary (Unknown Graph) ===")
    print(f"Total edges checked: {len(df_edge_unknown)}")
    print(f"Identifiable (True): {true_count_unknown}")
    print(f"Not identifiable (False): {false_count_unknown}")
else:
    print("\nNo edge-level unknown graph results to summarize.")


 Improved estimation


In [ ]:
# Full runnable script: builds lagged data, maps DAG to your columns, runs Graphical RICA (avg) and Cross-Moment,
# saves results and plots bar chart + DAG highlight.

import math
import warnings
import numpy as np
import pandas as pd
import torch
import networkx as nx
import matplotlib.pyplot as plt
from torch.nn import Parameter

# ---------------------------
# Utilities
# ---------------------------
def count_lists(mask):
    mask_arr = mask.cpu().numpy() if torch.is_tensor(mask) else np.array(mask)
    idx1 = 0
    idx0 = 0
    c_list = np.zeros(len(mask_arr), dtype=int)
    for i, m in enumerate(mask_arr):
        if m == 1:
            c_list[i] = idx1
            idx1 += 1
        else:
            c_list[i] = idx0
            idx0 += 1
    return c_list

def get_ratio_(Z, D, deg=2):
    var_u = np.mean(Z*D)
    sign = np.sign(var_u)
    diff_normal_D = np.mean(D**(deg)*Z) - deg*var_u*np.mean(D**(deg-1))
    diff_normal_Z = np.mean(Z**(deg)*D) - deg*var_u*np.mean(Z**(deg-1))
    alpha_sq = ((diff_normal_D) / (diff_normal_Z))
    if alpha_sq < 0:
        alpha_sq = -(abs(alpha_sq)**(1/(deg-1)))
    else:
        alpha_sq = alpha_sq**(1/(deg-1))
    alpha_sq = abs(alpha_sq)*sign
    return alpha_sq

def cross_moment(Z, D, Y, deg=2, tol=1e-8, max_deg=20):
    """
    Cross-Moment method (improved version, avoids infinite loop)

    Parameters:
        - Z (np.array): proxy variable observations
        - D (np.array): treatment observations
        - Y (np.array): outcome observations
        - deg (int): starting moment
        - tol (float): tolerance for denominator
        - max_deg (int): maximum number of iterations before giving up
    """
    denominator = 0
    numerator = np.nan
    tried = 0

    while abs(denominator) < tol and tried < max_deg:
        alpha_sq = get_ratio_(Z, D, deg)   # your ratio function
        numerator = np.mean(D * Y) - alpha_sq * np.mean(Y * Z)
        denominator = np.mean(D * D) - alpha_sq * np.mean(D * Z)

        deg += 1
        tried += 1

    if abs(denominator) < tol:
        # If denominator is still too small, return NaN instead of failing
        return np.nan

    return numerator / denominator


def cross_moment_sensor_fusion(W, Z, D, Y, deg=2, n_iter=100, percentage=0.9):
    length = len(W)
    sample_set = np.arange(length)
    n_samples = math.ceil(percentage*length)
    betas_est1 = np.zeros(n_iter)
    betas_est2 = np.zeros(n_iter)
    for it in range(n_iter):
        args_id = np.random.choice(sample_set, size=n_samples)
        W_tmp = W[args_id]
        Z_tmp = Z[args_id]
        D_tmp = D[args_id]
        Y_tmp = Y[args_id]
        beta_est1 = cross_moment(Z_tmp, D_tmp, Y_tmp, deg=deg)
        beta_est2 = cross_moment(W_tmp, D_tmp, Y_tmp, deg=deg)
        betas_est1[it] = beta_est1
        betas_est2[it] = beta_est2
    beta_est1 = np.mean(betas_est1)
    beta_est2 = np.mean(betas_est2)
    std1 = np.std(betas_est1)
    if std1==0:
        return beta_est1
    std2 = np.std(betas_est2)
    if std2==0:
        return beta_est2
    coef1 = 1./std1**2
    coef2 = 1./std2**2
    beta = (coef1*beta_est1 + coef2*beta_est2) / (coef1 + coef2)
    return beta


# ---------------------------
# Graphical RICA
# ---------------------------
def graphical_rica(latent, observed, g, data, data_whitening, epochs, lr, W_w,
                   w_init="random", w_true=None, momentum=0.0, lmbda=0.0,
                   use_scheduler=False, device=None):
    """

    Parameters
    ----------
    latent : int
        Number of latent nodes (assumed to be the first `latent` nodes in g.nodes() ordering).
    observed : int
        Number of observed nodes.
    g : nx.DiGraph
        Graph (nodes can be strings). We map nodes -> integer indices internally.
    data : torch.Tensor
        Original (centered) data (n_samples x observed).
    data_whitening : torch.Tensor
        Whitening matrix applied to B (same as W_w in your script).
    epochs : int
    lr : float
    W_w : torch.Tensor
        Whitening / re-scaling matrix (used in reference as W_with).
    w_init : str
        'random' or 'true' (cov_guess not used here).
    w_true : torch.Tensor or None
        True weights if available (only used to compute w_loss_data shape).
    momentum : float
    lmbda : float
        Regularization weight for reconstruction loss.
    use_scheduler : bool
        Whether to use a CosineAnnealingLR scheduler (like your reference).
    device : torch.device or None
        If provided, tensors are moved to the device.

    Returns
    -------
    loss_data : torch.Tensor (epochs,)
    w_loss_data : torch.Tensor (n_weights, epochs) or zeros if w_true None
    weight_pred : numpy.ndarray (n_weights,)
    weight_true : torch.Tensor or None
    """
    # map nodes to integer indices (preserve graph.nodes() ordering)
    nodes = list(g.nodes())
    node_to_idx = {n: i for i, n in enumerate(nodes)}
    edges = list(g.edges())
    n_weights = len(edges)

    if device is None:
        device = torch.device("cpu")

    data = data.to(device)
    data_whitening = data_whitening.to(device)
    W_w = W_w.to(device) if W_w is not None else torch.eye(observed, device=device)

    loss_data = torch.zeros(epochs, device=device)
    w_loss_data = torch.zeros(n_weights, epochs, device=device) if w_true is not None else torch.zeros(0, epochs, device=device)

    # initialize weight parameter (one scalar per edge)
    if w_init == "true" and w_true is not None:
        weight = Parameter(torch.clone(w_true).detach().to(device).requires_grad_(True))
        weight_true = w_true.to(device)
    else:
        # random init similar to your reference
        weight = Parameter(torch.Tensor(n_weights).normal_(0, 1).to(device))
        weight_true = w_true.to(device) if (w_true is not None) else None

    optimizer = torch.optim.RMSprop([weight], lr=lr, momentum=momentum)
    if use_scheduler:
        try:
            from torch.optim.lr_scheduler import CosineAnnealingLR
            scheduler = CosineAnnealingLR(optimizer, T_max=200)
        except Exception:
            scheduler = None
    else:
        scheduler = None

    min_loss = None
    weight_pred = None

    n_nodes = len(nodes)
    eye = torch.eye(n_nodes, device=device)

    for epoch in range(epochs):
        # build adjacency matrix (I on diagonal, -w on edges)
        adj = eye.clone()

        # Fill adj using node_to_idx mapping and weight vector
        # adj[src_idx, dst_idx] = - weight[idx_of_edge]
        for wi, e in enumerate(edges):
            src, dst = e
            src_idx = node_to_idx[src]
            dst_idx = node_to_idx[dst]
            # assign -weight scalar
            try:
                adj[src_idx, dst_idx] = -weight[wi]
            except Exception:
                # in case weight is not the correct shape, fallback to zero
                adj[src_idx, dst_idx] = -weight[wi]

        # compute B = inv(adj).T
        try:
            B = torch.inverse(adj).t()
        except RuntimeError:
            # add small diagonal jitter to help invertibility
            jitter = 1e-6 * torch.eye(adj.size(0), device=device)
            B = torch.inverse(adj + jitter).t()

        # slice B rows corresponding to observed variables
        # Assuming latent nodes are first `latent` entries in nodes ordering
        # and observed correspond to indices latent .. latent+observed-1
        start = latent
        end = latent + observed
        B_obs = B[start:end, :]        # shape: observed x n_nodes
        B_obs = W_w.matmul(B_obs)      # apply whitening (W_w @ B_obs)

        # choose pseudoinv or direct use for estimating latents
        # keep behavior consistent with your reference: BB = B_obs or pseudoinv
        BB = B_obs  # in your reference pseudoinv optional; we keep direct B_obs

        # compute latents for current batch (here we use full data)
        # data_whitening shape: n_samples x observed
        latents = data_whitening.matmul(BB)  # shape: n_samples x n_nodes

        # Latent loss: L1 on latents (sparsity), same as reference
        loss_latent = latents.abs().mean()

        # Reconstruction loss (optional)
        loss_recon = 0.0
        if lmbda != 0:
            output = latents.matmul(BB.t())
            diff = output - data_whitening
            loss_recon = (diff * diff).mean()

        loss = lmbda * loss_recon + loss_latent

        loss_data[epoch] = loss.detach()

        # keep best weights
        if min_loss is None or loss_data[epoch].item() < min_loss:
            min_loss = loss_data[epoch].item()
            weight_pred = weight.detach().cpu().numpy().copy()

        # track weight differences to true if available
        if weight_true is not None and w_loss_data.numel() > 0:
            # try to expand/align shapes
            try:
                w_loss_data[:, epoch] = (weight.detach() - weight_true.detach()).abs().cpu()
            except Exception:
                # skip if shapes mismatch
                pass

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if scheduler is not None:
            scheduler.step()

    # final fallback if weight_pred was never set
    if weight_pred is None:
        try:
            weight_pred = weight.detach().cpu().numpy().copy()
        except Exception:
            weight_pred = np.zeros(n_weights)

    return loss_data.cpu(), w_loss_data.cpu(), weight_pred, (weight_true.cpu() if weight_true is not None else None)


# ---------------------------
# 1. Load CSV and create lagged columns
# ---------------------------
CSV_PATH = "causal_feature_matrix.csv"  # adjust path
df_raw = pd.read_csv(CSV_PATH, parse_dates=["Date"], index_col="Date")
df = df_raw.copy()

# create lagged columns
for col in df.columns:
    df[f"{col}_t"] = df[col].values
    df[f"{col}_t-1"] = df[col].shift(1).values
df = df.dropna()

train_frac = 0.8
n_samples = df.shape[0]
train_idx = np.random.choice(n_samples, size=int(train_frac * n_samples), replace=False)
df_train = df.iloc[train_idx].copy()

# ---------------------------
# 2. Build DAG mapping
# ---------------------------
def find_best_column(base, names):
    base_low = base.lower()
    for c in names:
        if c.lower().startswith(base_low) or base_low in c.lower():
            return c
    return None

def build_mapped_dag(df_columns):
    G = nx.DiGraph()
    latent_nodes = ["L_micro_t-1", "L_sentiment_t-1"]
    canonical_observed = ["USDX", "IRX", "TNX", "VIX", "SP500", "XLK", "XLF", "XLE"]
    canonical_proxies = ["OpenInterest", "PricingError", "SPV"]
    sector_vol = [f"{s}_Volume" for s in ["XLK","XLF","XLE"]]
    sector_ami = [f"{s}_Amihud" for s in ["XLK","XLF","XLE"]]
    canonical_proxies += sector_vol + sector_ami

    mapping = {}
    observed_mapped = []
    proxies_mapped = []

    for base in canonical_observed:
        found = find_best_column(base, df_columns)
        if found:
            mapping[base] = found.replace("_t","").replace("_t-1","")
            observed_mapped.append(mapping[base])
    for base in canonical_proxies:
        found = find_best_column(base, df_columns)
        if found:
            mapping[base] = found.replace("_t","").replace("_t-1","")
            proxies_mapped.append(mapping[base])

    # add nodes
    nodes = latent_nodes[:]
    for nm in observed_mapped + proxies_mapped:
        nodes += [f"{nm}_t-1", f"{nm}_t"]
    G.add_nodes_from(nodes)

    # latent to observed
    if "IRX" in mapping: G.add_edge("L_micro_t-1", f"{mapping['IRX']}_t")
    if "TNX" in mapping: G.add_edge("L_micro_t-1", f"{mapping['TNX']}_t")
    if "USDX" in mapping: G.add_edge("L_micro_t-1", f"{mapping['USDX']}_t")
    if "VIX" in mapping: G.add_edge("L_sentiment_t-1", f"{mapping['VIX']}_t")

    # proxy edges
    pairs = [("OpenInterest","SP500"), ("PricingError","OpenInterest"), ("SPV","SP500")]
    latent_counter = 1
    for s,t in pairs:
        if s in mapping and t in mapping:
            G.add_edge(f"{mapping[s]}_t-1", f"{mapping[t]}_t")
            ln = f"L_p{latent_counter}"
            latent_counter += 1
            G.add_node(ln)
            G.add_edge(ln, f"{mapping[t]}_t")

    # lag edges
    for nm in observed_mapped + proxies_mapped:
        if f"{nm}_t-1" in G.nodes() and f"{nm}_t" in G.nodes():
            G.add_edge(f"{nm}_t-1", f"{nm}_t")

    return G, mapping, observed_mapped, proxies_mapped, latent_nodes

G, mapping, observed_mapped, proxies_mapped, latent_nodes = build_mapped_dag(df.columns)

# ---------------------------
# 3. Prepare data for RICA
# ---------------------------
observed_for_rica = []
for c in observed_mapped:
    observed_for_rica += [f"{c}_t-1", f"{c}_t"]

X = torch.tensor(df_train[observed_for_rica].values, dtype=torch.float32)
X_centered = X - X.mean(0)
cov = torch.cov(X_centered.T)
U, S, V = torch.svd(cov)
W_whitening = torch.diag(1.0 / torch.sqrt(S)) @ V.T
X_whitened = X_centered @ W_whitening.T


# ---------------------------
# 4. Run Graphical RICA multiple times
# ---------------------------
def run_graphical_rica_avg(G, latent_count, observed_count, X_tensor, X_whitened,
                            n_runs=3, epochs=200, lr=0.05, device=None):
    edges = list(G.edges())
    all_weights = []
    for i in range(n_runs):
        loss_data, w_loss_data, weight_pred, weight_true = graphical_rica(
            latent=latent_count,
            observed=observed_count,
            g=G,
            data=X_tensor,
            data_whitening=X_whitened,
            epochs=epochs,
            lr=lr,
            W_w=W_whitening,
            w_init="random",
            w_true=torch.zeros(len(edges)),
            momentum=0,
            lmbda=0.0,
            use_scheduler=False,
            device=device
        )
        # ensure numpy array
        if isinstance(weight_pred, np.ndarray):
            all_weights.append(weight_pred)
        else:
            try:
                all_weights.append(np.array(weight_pred))
            except Exception:
                all_weights.append(np.zeros(len(edges)))

    avg = np.mean(all_weights, axis=0)
    rica_effects = {e: w for e, w in zip(edges, avg)}
    return edges, avg, all_weights,rica_effects


# ---------------------------
# 4b. Run Graphical RICA multiple times
# ---------------------------
latent_count = len(latent_nodes)
observed_count = len(observed_for_rica)
edges, avg_weights, all_weights, rica_effects = run_graphical_rica_avg(
    G,
    latent_count,
    observed_count,
    X,
    X_whitened,
    n_runs=3,
    epochs=200,
    lr=0.05,
    device=None
)


# ---------------------------
# 5. Cross-Moment for SP500
# ---------------------------
sp500_base = mapping.get("SP500", None)
sp500_node = f"{sp500_base}_t" if sp500_base else None

cross_effects = {}
all_t_cols = [c for c in df.columns if c.endswith("_t")]
if sp500_node is None:
    warnings.warn("No SP500 target detected. Skipping Cross-Moment.")
else:
    proxy_pool = [c for c in all_t_cols if c != sp500_node]
    for D_col in all_t_cols:
        if D_col == sp500_node:
            continue
        # proxies: all variables except D and target
        Z_candidates = [z for z in proxy_pool if z != D_col]
        if len(Z_candidates) < 2:
            # not enough proxies → skip
            cross_effects[D_col] = np.nan
            continue

        # take the first two proxies (can be extended to more later)
        W = df_train[Z_candidates[0]].values
        Z = df_train[Z_candidates[1]].values
        D = df_train[D_col].values
        Y = df_train[sp500_node].values

        try:
            beta = cross_moment_sensor_fusion(
                W, Z, D, Y, deg=2, n_iter=100, percentage=0.9
            )
            cross_effects[D_col] = beta
        except Exception:
            cross_effects[D_col] = np.nan

# ---------------------------
# 6. Merge results
# ---------------------------
sources = sorted(set([e[0] for e in edges] + list(cross_effects.keys())))
rows = []
for s in sources:
    # sum weights of edges starting from s
    weight_sum = sum([rica_effects[e] for e in edges if e[0]==s])
    rows.append({"Source": s, "GraphicalRICA": weight_sum, "CrossMoment": cross_effects.get(s, np.nan)})
df_results = pd.DataFrame(rows).sort_values("GraphicalRICA", ascending=False).reset_index(drop=True)
out_csv = "sp500_effects_comparison.csv"
df_results.to_csv(out_csv, index=False)
print(df_results)
print("Saved results to", out_csv)

# ---------------------------
# 7. Plot
# ---------------------------
plt.figure(figsize=(12,6))
df_plot = df_results.set_index("Source")
ax = df_plot.plot(kind="bar", figsize=(12,6))
ax.set_ylabel("Estimated Effect")
ax.set_title("Graphical RICA & Cross-Moment")
plt.tight_layout()
plt.show()

print("Done.")

Prediction

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score, mean_squared_error, r2_score
import matplotlib.pyplot as plt

# -----------------------------
# 1. Load LiNGAM weights
# -----------------------------
coef_df = pd.read_csv("sp500_effects_comparison.csv")
coef_df = coef_df.replace(["", "None", "nan"], np.nan)
coef_df["Coefficient"] = coef_df[["GraphicalRICA", "CrossMoment"]].mean(axis=1, skipna=True)
coef_df = coef_df.dropna(subset=["Coefficient"])

# -----------------------------
# 2. Load full dataset
# -----------------------------
data = pd.read_csv("causal_feature_matrix.csv", index_col=0, parse_dates=True)

# -----------------------------
# 3. Build lagged/current features
# -----------------------------
X = pd.DataFrame(index=data.index)
beta_dict = {}

for _, row in coef_df.iterrows():
    source = row["Source"]
    coef = row["Coefficient"]
    if source.endswith("_t-1"):
        base_name = source.replace("_t-1", "")
        if base_name in data.columns:
            new_col = f"{base_name}_lag1"
            X[new_col] = data[base_name].shift(1)
            beta_dict[new_col] = coef
    elif source.endswith("_t"):
        base_name = source.replace("_t", "")
        if base_name in data.columns:
            X[base_name] = data[base_name]
            beta_dict[base_name] = coef

# Drop rows with NaN due to lag
X = X.dropna()
y = data["SP500_Ret"].loc[X.index]

# -----------------------------
# 4. Split last 20% as test
# -----------------------------
n_test = int(len(X) * 0.2)
X_test = X.iloc[-n_test:]
y_test = y.iloc[-n_test:]

# -----------------------------
# 5. Predict using LiNGAM weights
# -----------------------------
aligned_beta = np.array([beta_dict.get(col, 0.0) for col in X_test.columns])
y_pred_cont = X_test.values @ aligned_beta  # continuous prediction

# -----------------------------
# 6. Convert to classification
# -----------------------------
y_test_class = (y_test > 0).astype(int)
y_pred_class = (y_pred_cont > 0).astype(int)

# -----------------------------
# 7. Evaluation
# -----------------------------
# Continuous metrics
mse = mean_squared_error(y_test, y_pred_cont)
r2 = r2_score(y_test, y_pred_cont)

# Directional accuracy
directional_accuracy = np.mean(np.sign(y_test) == np.sign(y_pred_cont))

# Classification metrics
accuracy = accuracy_score(y_test_class, y_pred_class)

# For AUC, map continuous prediction to probability using sigmoid
y_pred_prob = 1 / (1 + np.exp(-y_pred_cont))  # simple mapping
auc_score = roc_auc_score(y_test_class, y_pred_prob)

print("\n=== Model Performance ===")
print(f"Mean Squared Error: {mse:.12f}")
print(f"R^2 Score: {r2:.12f}")
print(f"AUC Score: {auc_score:.4f}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Directional Accuracy: {directional_accuracy:.4f}")


Comparison Analysis

In [ ]:
import pandas as pd
import numpy as np
import os
import yfinance as yf
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, mean_squared_error, r2_score

# ==========================================================
# 1) Download data
# ==========================================================
def download_data(tickers, start, end):
    data = {}
    for ticker in tickers:
        try:
            df = yf.download(ticker, start=start, end=end, progress=False)
            if not df.empty and 'Close' in df.columns:
                df['LogRet'] = np.log(df['Close'] / df['Close'].shift(1))
                df = df[['LogRet']].dropna()
                if not df.empty:
                    data[ticker] = df
                else:
                    print(f"[warn] {ticker}: no valid log return")
            else:
                print(f"[warn] {ticker}: empty or no Close column")
        except Exception as e:
            print(f"[error] {ticker}: {e}")
    return data

start_date = '2020-09-01'
end_date = '2025-09-01'

stock_list = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA', 'BRK-B', 'UNH', 'JNJ',
        'JPM', 'V', 'PG', 'XOM', 'HD', 'CVX', 'MA', 'BAC', 'ABBV', 'PFE',
        'AVGO', 'LLY', 'KO', 'WMT', 'DIS', 'NFLX', 'ADBE', 'CRM', 'CSCO', 'PEP',
        'TMO', 'ABT', 'DHR', 'ACN', 'VZ', 'NKE', 'CMCSA', 'T', 'COST', 'WFC',
        'MRK', 'PM', 'UPS', 'RTX', 'TXN', 'QCOM', 'SBUX', 'SPGI', 'BLK', 'AMGN']
index_list = ['^N225', '^HSI', '^GDAXI', '^FTSE', '^DJI', '^GSPC']

stock_data = download_data(stock_list, start_date, end_date)
index_data = download_data(index_list, start_date, end_date)

# ==========================================================
# 2) Merge returns
# ==========================================================
def merge_returns(data_dict):
    df_list = []
    for name, df in data_dict.items():
        df = df.rename(columns={'LogRet': name})
        df_list.append(df)
    if not df_list:
        raise RuntimeError("No valid data to merge.")
    merged = pd.concat(df_list, axis=1).fillna(method='ffill')
    return merged.dropna()

stock_return_df = merge_returns(stock_data)
index_return_df = merge_returns(index_data)


# ==========================================================
# 3) Create lag features - MORE SOPHISTICATED LAG STRUCTURE
# ==========================================================
def build_lag_features(df, short_lags=[1, 2, 3], medium_lags=[5, 10], long_lags=[20, 50]):
    feat_df = pd.DataFrame(index=df.index)
    for col in df.columns:
        # Short-term lags (recent days)
        for lag in short_lags:
            feat_df[f"{col}_lag_{lag}"] = df[col].shift(lag)

        # Medium-term lags
        for lag in medium_lags:
            feat_df[f"{col}_lag_{lag}"] = df[col].shift(lag)

        # Long-term lags
        for lag in long_lags:
            feat_df[f"{col}_lag_{lag}"] = df[col].shift(lag)

        # Rolling statistics as additional features
        feat_df[f"{col}_rolling_mean_5"] = df[col].rolling(5).mean().shift(1)
        feat_df[f"{col}_rolling_std_5"] = df[col].rolling(5).std().shift(1)
        feat_df[f"{col}_rolling_mean_10"] = df[col].rolling(10).mean().shift(1)

    return feat_df

# Use different lag structures for indices vs stocks
index_feats = build_lag_features(index_return_df,
                               short_lags=[1, 2, 3, 4, 5],
                               medium_lags=[10, 15],
                               long_lags=[])

stock_feats = build_lag_features(stock_return_df,
                               short_lags=[1, 2, 3],
                               medium_lags=[5, 10, 20],
                               long_lags=[50])

# ==========================================================
# 4) Concatenate and align with target
# ==========================================================
all_feats = pd.concat([index_feats, stock_feats], axis=1)

preferred_index = '^GSPC'
fallback_stock = 'AAPL'

if preferred_index in index_return_df.columns:
    raw_target_series = index_return_df[preferred_index]
    print(f"Using {preferred_index} as target (index).")
elif fallback_stock in stock_return_df.columns:
    raw_target_series = stock_return_df[fallback_stock]
    print(f"Using fallback stock {fallback_stock} as target.")
else:
    raw_target_series = stock_return_df.iloc[:, 0]
    print(f"Using first available stock {raw_target_series.name} as target (fallback).")

target_next = raw_target_series.shift(-1)

# align feature index with target_next
common_idx = all_feats.index.intersection(target_next.index)
all_feats = all_feats.loc[common_idx]
target_next = target_next.loc[common_idx]

# drop rows with NaN in any lag or target
all_feats['target'] = target_next
all_feats.dropna(inplace=True)
target_next = all_feats['target']
all_feats = all_feats.drop('target', axis=1)

# classification target
y_class = (target_next > 0).astype(int)

# ==========================================================
# 5) Train/test split (time-based)
# ==========================================================
n_samples = len(all_feats)
split_point = int(n_samples * 0.8)

X = all_feats.values
y = y_class.values

X_train, X_test = X[:split_point], X[split_point:]
y_train, y_test = y[:split_point], y[split_point:]

print(f"Train samples: {len(X_train)}, Test samples: {len(X_test)}")

# scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ==========================================================
# 6) Train logistic regression
# ==========================================================
clf = LogisticRegression(penalty='l1', C=0.05, solver='liblinear', max_iter=1000)
clf.fit(X_train_scaled, y_train)

y_prob = clf.predict_proba(X_test_scaled)[:, 1]
y_pred = (y_prob > 0.5).astype(int)

# ==========================================================
# 7) Evaluation
# ==========================================================
mse = mean_squared_error(y_test, y_prob)
r2 = r2_score(y_test, y_prob)
direction_true = np.sign(y_test)
direction_pred = np.sign(y_prob)
directional_accuracy = np.mean(direction_true == direction_pred)

auc_score = roc_auc_score(y_test, y_prob)
accuracy = accuracy_score(y_test, y_pred)

print("\n=== Model Performance ===")
print(f"Mean Squared Error: {mse:.12f}")
print(f"R^2 Score: {r2:.12f}")
print(f"AUC Score: {auc_score:.4f}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Directional Accuracy: {directional_accuracy:.4f}")
